# P3a -- Label-efficiency curves, land-cover task (25%/50% budgets)

**Run this on Kaggle** (Settings -> Accelerator -> GPU T4 x2 or P100; Settings -> Internet -> On).
Continuation of P0/P0b (both done 2026-10-02, real 100%-budget results already
verified and reused here, not re-run). Repeats H7's original label-efficiency idea
(25/50/100% budgets) with the now-verified recipes from P0/P0b, for the land-cover
task (Prithvi, U-Net, Clay). See
`Geospatial AI for Public Health -- AI Paper Track (Phases P0-P6) -- Plan.md` for the
full track. Flood/SAR label efficiency is a separate notebook (P3b) -- Prithvi is
excluded there (verified inapplicable in P2: no SAR-native Sen1Floods11 checkpoint
exists publicly).

**Design:** same AOI/labels/held-out block as P0/P0b (64 train tiles, 16 held-out).
25%/50% budgets are **nested prefixes of a single fixed-seed shuffle** of the 64 train
tiles (16 and 32 tiles respectively) -- so the 50% budget's data is a superset of the
25% budget's, a real "adding more data" curve, not three independent random draws.
100% reuses P0/P0b's actual already-verified numbers rather than re-running a case
already on record.

**Robustness:** each (model, budget) run is wrapped so a late failure doesn't lose
earlier results -- the comparison CSV is written incrementally after every single run,
not only at the end.


## 1. Setup

In [ ]:
!pip install -q "numpy>=2.2" terratorch==1.2.13 lightning segmentation-models-pytorch pystac-client planetary-computer stackstac rioxarray huggingface_hub timm einops torchmetrics
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"

!rm -rf repo && git clone -q https://github.com/khalilurrrahmanridoykhan/geohealth-risk-mapping.git repo
!rm -rf clay_repo && git clone -q https://github.com/Clay-foundation/model.git clay_repo
import sys
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')


**Running the real pipeline as a subprocess, not inline** -- same verified reason as
every earlier phase in this track: terratorch needs numpy>=2.2, Kaggle's kernel
process already has an older numpy loaded in memory from its own startup, and a
subprocess sidesteps the resulting stale-vs-fresh mismatch entirely.

## 2. Write the real pipeline to a script (runs as a subprocess, see note above)

In [ ]:
%%writefile run_p3a.py
import sys, json, traceback
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')

import numpy as np
import torch
import torch.nn.functional as F
import lightning.pytorch as pl
import segmentation_models_pytorch as smp
import terratorch  # noqa: F401
from torch.utils.data import DataLoader, Dataset
import pandas as pd

from src.imagery import get_imagery
from src.landcover import reclassify_worldcover, WATER, BUILT_UP
from src.tiling import tile_array, assign_tile_split

from claymodel.finetune.segment.factory import Segmentor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert DEVICE == 'cuda', 'this notebook needs a GPU runtime -- check Settings > Accelerator'

CLASS_NAMES = ['other', 'water', 'built_up']
TILE_SIZE = 256
PRITHVI_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']
AOI = (90.30, 23.75, 90.55, 23.95)  # same Dhaka-area tile as P0/P0b

# ---- same AOI/labels/split as P0/P0b, exactly ----
composite, meta = get_imagery(AOI, '2026-01-01/2026-02-28', sensor='sentinel-2',
                               cloud_cover_max=60, max_scenes=1, resolution=10,
                               bands=PRITHVI_BANDS)
catalog_bbox = AOI
import pystac_client, planetary_computer, stackstac
catalog = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1', modifier=planetary_computer.sign_inplace)
wc_search = catalog.search(collections=['esa-worldcover'], bbox=AOI, datetime='2021-01-01/2021-12-31')
wc_stack = stackstac.stack(list(wc_search.items()), assets=['map'], bounds_latlon=AOI, resolution=10, epsg=meta['epsg'])
worldcover = wc_stack.compute().isel(time=0, band=0).values
labels_4class = reclassify_worldcover(worldcover)
label_map = np.zeros_like(labels_4class, dtype='int64')
label_map[labels_4class == WATER] = 1
label_map[labels_4class == BUILT_UP] = 2

feat_tiles, coords = tile_array(np.nan_to_num(composite.values), tile_size=TILE_SIZE)
label_tiles, _ = tile_array(label_map, tile_size=TILE_SIZE)
test_flags = assign_tile_split(coords, width=composite.shape[-1], test_fraction=0.3)
print(f'{len(feat_tiles)} tiles total: {sum(not f for f in test_flags)} train, {sum(test_flags)} held-out')

all_train_feats = [f for f, t in zip(feat_tiles, test_flags) if not t]
all_train_labels = [l for l, t in zip(label_tiles, test_flags) if not t]
test_feats = [f for f, t in zip(feat_tiles, test_flags) if t]
test_labels = [l for l, t in zip(label_tiles, test_flags) if t]
N_TRAIN = len(all_train_feats)
print(f'{N_TRAIN} train tiles available, {len(test_feats)} held-out test tiles')

# Fixed-seed shuffle -> nested prefixes for 25%/50% budgets (50% is a superset of 25%).
rng = np.random.RandomState(42)
shuffled_idx = rng.permutation(N_TRAIN)

results = []

def save_results():
    pd.DataFrame(results).to_csv('comparison_p3a.csv', index=False)
    print('--- comparison_p3a.csv so far ---')
    print(pd.DataFrame(results).to_string(index=False))

def tidy_terratorch(metrics, model_name, budget):
    row = {'model': model_name, 'budget': budget}
    for k, v in metrics[0].items():
        if 'iou' in k.lower():
            name = k.split('/')[-1].lower()
            name = name[4:] if name.startswith('iou_') else name
            row[name] = round(v, 3)
    return row

# Already-verified 100% results from P0/P0b -- not re-run.
results.append({'model': 'prithvi_eo_v2_300_tl', 'budget': 1.0, 'other': 0.934, 'water': 0.442, 'built_up': 0.284})
results.append({'model': 'unet_resnet34_scratch', 'budget': 1.0, 'other': 0.929, 'water': 0.325, 'built_up': 0.201})
results.append({'model': 'clay_v1.5', 'budget': 1.0, 'other': 0.936, 'water': 0.345, 'built_up': 0.224})
save_results()

CONSTANT_SCALE = 1e-4
CLAY_MEAN = np.array([1105., 1355., 1552., 2743., 2388., 1835.], dtype='float32')
CLAY_STD = np.array([1809., 1757., 1888., 1742., 1470., 1379.], dtype='float32')
CLAY_WAVES = torch.tensor([0.493, 0.56, 0.665, 0.842, 1.61, 2.19], dtype=torch.float32)
CLAY_GSD = torch.tensor(10.0)


def run_prithvi(train_feats, train_labels, budget):
    band_mean = (np.stack(train_feats) * CONSTANT_SCALE).mean(axis=(0, 2, 3))
    band_std = (np.stack(train_feats) * CONSTANT_SCALE).std(axis=(0, 2, 3)) + 1e-6

    class PDS(Dataset):
        def __init__(self, feats, labels):
            self.feats, self.labels = feats, labels
        def __len__(self):
            return len(self.feats)
        def __getitem__(self, idx):
            x = (self.feats[idx] * CONSTANT_SCALE - band_mean[:, None, None]) / band_std[:, None, None]
            return {'image': torch.from_numpy(x.astype('float32')), 'mask': torch.from_numpy(self.labels[idx])}

    train_loader = DataLoader(PDS(train_feats, train_labels), batch_size=8, shuffle=True, num_workers=2)
    test_loader = DataLoader(PDS(test_feats, test_labels), batch_size=8, num_workers=2)

    model = terratorch.tasks.SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory',
        model_args={
            'backbone': 'prithvi_eo_v2_300_tl', 'backbone_pretrained': True,
            'backbone_img_size': TILE_SIZE,
            'backbone_bands': ['BLUE', 'GREEN', 'RED', 'NIR_BROAD', 'SWIR_1', 'SWIR_2'],
            'backbone_coords_encoding': [],
            'necks': [{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]},
                      {'name': 'ReshapeTokensToImage'}, {'name': 'LearnedInterpolateToPyramidal'}],
            'decoder': 'UNetDecoder', 'decoder_channels': [512, 256, 128, 64],
            'head_dropout': 0.1, 'num_classes': 3,
        },
        loss='dice', optimizer='AdamW', lr=1e-4, optimizer_hparams={'weight_decay': 0.1},
        scheduler='ReduceLROnPlateau', scheduler_hparams={'factor': 0.5, 'patience': 5},
        freeze_backbone=False, freeze_decoder=False, class_names=CLASS_NAMES, plot_on_val=False,
    )
    trainer = pl.Trainer(accelerator='gpu', devices=1, precision='16-mixed', max_epochs=100,
                          log_every_n_steps=5, callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
                          default_root_dir=f'output/p3a_prithvi_{budget}/')
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)
    metrics = trainer.test(model, dataloaders=test_loader)
    return tidy_terratorch(metrics, 'prithvi_eo_v2_300_tl', budget)


def run_unet(train_feats, train_labels, budget):
    band_mean = (np.stack(train_feats) * CONSTANT_SCALE).mean(axis=(0, 2, 3))
    band_std = (np.stack(train_feats) * CONSTANT_SCALE).std(axis=(0, 2, 3)) + 1e-6

    class UDS(Dataset):
        def __init__(self, feats, labels):
            self.feats, self.labels = feats, labels
        def __len__(self):
            return len(self.feats)
        def __getitem__(self, idx):
            x = (self.feats[idx] * CONSTANT_SCALE - band_mean[:, None, None]) / band_std[:, None, None]
            return {'image': torch.from_numpy(x.astype('float32')), 'mask': torch.from_numpy(self.labels[idx])}

    train_loader = DataLoader(UDS(train_feats, train_labels), batch_size=8, shuffle=True, num_workers=2)
    test_loader = DataLoader(UDS(test_feats, test_labels), batch_size=8, num_workers=2)

    class UNetBaseline(pl.LightningModule):
        def __init__(self, in_channels, num_classes):
            super().__init__()
            self.model = smp.Unet(encoder_name='resnet34', encoder_weights=None, in_channels=in_channels, classes=num_classes)
            self.dice = smp.losses.DiceLoss(mode='multiclass')
        def forward(self, x):
            return self.model(x)
        def _step(self, batch, stage):
            logits = self(batch['image'])
            loss = self.dice(logits, batch['mask'])
            self.log(f'{stage}/loss', loss, prog_bar=(stage != 'train'))
            return logits, loss
        def training_step(self, batch, _):
            _, loss = self._step(batch, 'train')
            return loss
        def validation_step(self, batch, _):
            self._step(batch, 'val')
        def test_step(self, batch, _):
            logits, _ = self._step(batch, 'test')
            preds = logits.argmax(dim=1)
            for c, name in enumerate(CLASS_NAMES):
                p, t = preds == c, batch['mask'] == c
                union = (p | t).sum().float()
                iou = (p & t).sum().float() / union if union > 0 else torch.tensor(1.0, device=p.device)
                self.log(f'test/iou_{name}', iou)
        def configure_optimizers(self):
            optimizer = torch.optim.AdamW(self.parameters(), lr=1e-4, weight_decay=0.1)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)
            return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'monitor': 'val/loss'}}

    model = UNetBaseline(in_channels=len(PRITHVI_BANDS), num_classes=3)
    trainer = pl.Trainer(accelerator='gpu', devices=1, precision='16-mixed', max_epochs=100,
                          log_every_n_steps=5, callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
                          default_root_dir=f'output/p3a_unet_{budget}/')
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)
    metrics = trainer.test(model, dataloaders=test_loader)
    return tidy_terratorch(metrics, 'unet_resnet34_scratch', budget)


def run_clay(train_feats, train_labels, budget):
    class CDS(Dataset):
        def __init__(self, feats, labels):
            self.feats, self.labels = feats, labels
        def __len__(self):
            return len(self.feats)
        def __getitem__(self, idx):
            x = (self.feats[idx] - CLAY_MEAN[:, None, None]) / CLAY_STD[:, None, None]
            return {'pixels': torch.from_numpy(x.astype('float32')), 'label': torch.from_numpy(self.labels[idx]),
                    'time': torch.zeros(4), 'latlon': torch.zeros(4)}

    train_loader = DataLoader(CDS(train_feats, train_labels), batch_size=8, shuffle=True, num_workers=2)
    test_loader = DataLoader(CDS(test_feats, test_labels), batch_size=8, num_workers=2)

    import huggingface_hub
    ckpt_path = huggingface_hub.hf_hub_download(repo_id='made-with-clay/Clay', filename='v1.5/clay-v1.5.ckpt')

    class ClaySegmentor(pl.LightningModule):
        def __init__(self):
            super().__init__()
            self.model = Segmentor(num_classes=3, ckpt_path=None)
            self.model.encoder.load_from_ckpt(ckpt_path)
            self.loss_fn = smp.losses.FocalLoss(mode='multiclass')
            from torchmetrics.classification import MulticlassJaccardIndex
            self.iou_per_class = MulticlassJaccardIndex(num_classes=3, average='none')
        def forward(self, batch):
            return self.model({'pixels': batch['pixels'], 'time': batch['time'], 'latlon': batch['latlon'],
                                'gsd': CLAY_GSD, 'waves': CLAY_WAVES})
        def configure_optimizers(self):
            optimizer = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad],
                                           lr=1e-5, weight_decay=0.05, betas=(0.9, 0.95))
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=1, eta_min=1e-5 * 100)
            return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}
        def _shared_step(self, batch, stage):
            labels = batch['label'].long()
            outputs = self(batch)
            if outputs.shape[-2:] != labels.shape[-2:]:
                outputs = F.interpolate(outputs, size=labels.shape[-2:], mode='bilinear', align_corners=False)
            loss = self.loss_fn(outputs, labels)
            self.log(f'{stage}/loss', loss, on_epoch=True, prog_bar=(stage != 'train'))
            return outputs, labels, loss
        def training_step(self, batch, _):
            _, _, loss = self._shared_step(batch, 'train')
            return loss
        def validation_step(self, batch, _):
            self._shared_step(batch, 'val')
        def test_step(self, batch, _):
            outputs, labels, _ = self._shared_step(batch, 'test')
            per_class = self.iou_per_class(outputs, labels)
            for name, val in zip(CLASS_NAMES, per_class):
                self.log(f'test/iou_{name}', val, on_epoch=True)

    model = ClaySegmentor()
    trainer = pl.Trainer(accelerator='gpu', devices=1, precision='bf16-mixed', max_epochs=100,
                          log_every_n_steps=5, callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
                          default_root_dir=f'output/p3a_clay_{budget}/')
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)
    metrics = trainer.test(model, dataloaders=test_loader)
    return tidy_terratorch(metrics, 'clay_v1.5', budget)


for budget in [0.25, 0.50]:
    n = max(1, int(round(N_TRAIN * budget)))
    subset_idx = shuffled_idx[:n]
    sub_feats = [all_train_feats[i] for i in subset_idx]
    sub_labels = [all_train_labels[i] for i in subset_idx]
    print(f'=== budget {budget} -> {n}/{N_TRAIN} train tiles ===')

    for name, fn in [('prithvi', run_prithvi), ('unet', run_unet), ('clay', run_clay)]:
        try:
            print(f'--- running {name} @ budget {budget} ---')
            row = fn(sub_feats, sub_labels, budget)
            print(row)
            results.append(row)
        except Exception as e:
            print(f'FAILED: {name} @ budget {budget}: {e}')
            traceback.print_exc()
            results.append({'model': name, 'budget': budget, 'error': str(e)})
        save_results()

print('=== FINAL ===')
print(pd.DataFrame(results).to_string(index=False))


## 3. Run it as a subprocess, stream the result back

In [ ]:
import subprocess

proc = subprocess.run(['python', 'run_p3a.py'], capture_output=True, text=True)
print(proc.stdout[-25000:])
print('---stderr (tail)---')
print(proc.stderr[-10000:])
print('returncode:', proc.returncode)
# NOTE: unlike earlier phases, this does NOT assert returncode==0 -- a single
# (model, budget) failure is caught and logged inside run_p3a.py itself (see its
# try/except), so a partial comparison_p3a.csv with some real rows and some
# 'error' rows is a valid, honest outcome worth reading, not a reason to discard
# everything that did succeed.


## 4. Results

In [ ]:
import pandas as pd
comparison = pd.read_csv('comparison_p3a.csv')
print(comparison.to_string(index=False))


## 5. Honest notes for whoever runs this next

- **This cell sequence has not been executed anywhere yet.** It chains up to 6 full
  training runs (2 budgets x 3 models) in one Kaggle session, reusing P0/P0b's
  already-verified recipes -- each one is individually the same code already proven to
  work, but this is the first time they're run back-to-back in one process. GPU memory
  not being released between runs is a real risk worth watching for (each model/trainer
  goes out of scope at the end of its function, but PyTorch/CUDA caching could still
  accumulate) -- if a later run OOMs when earlier ones didn't, that is the first thing
  to check, not a mystery.
- **25%/50% budgets are nested prefixes of one fixed-seed shuffle** (seed 42) of the 64
  train tiles -- 16 and 32 tiles respectively, with 50%'s set a superset of 25%'s. This
  is a real, deliberate design choice (a genuine "more data helps" curve), not an
  oversight; an independent-random-draw-per-budget design was considered and rejected
  as less interpretable.
- **Each (model, budget) run is wrapped in try/except and the CSV is saved after every
  single one** -- if this notebook errors out partway through, whatever's already in
  `comparison_p3a.csv` (downloaded from this kernel's output even on a failed run) is
  real, usable, honest progress, not something to throw away and restart from scratch.
- Once this runs clean (or partially clean): bring back the actual numbers, including
  any 'error' rows, and compare the shape of these curves honestly against H7's
  original (CPU, under-trained) finding that Prithvi showed no working label-efficiency
  advantage at all -- the real question P3 asks is whether that was a real property of
  foundation models here, or an artifact of H7's own under-training, now that the
  recipes are verified.


**Update 2026-10-02/03, real result (Kaggle kernel version 1, COMPLETE, ~2+ hours,
T4 GPU, all 6 new runs succeeded with zero failures):**

| model | budget | other | water | built_up |
|---|---|---|---|---|
| prithvi_eo_v2_300_tl | 0.25 | 0.915 | 0.412 | 0.152 |
| prithvi_eo_v2_300_tl | 0.50 | 0.930 | 0.464 | 0.188 |
| prithvi_eo_v2_300_tl | 1.00 | 0.934 | 0.442 | 0.284 |
| unet_resnet34_scratch | 0.25 | 0.871 | 0.315 | 0.164 |
| unet_resnet34_scratch | 0.50 | 0.907 | 0.321 | 0.186 |
| unet_resnet34_scratch | 1.00 | 0.929 | 0.325 | 0.201 |
| clay_v1.5 | 0.25 | 0.905 | 0.345 | 0.248 |
| clay_v1.5 | 0.50 | 0.928 | 0.349 | 0.236 |
| clay_v1.5 | 1.00 | 0.936 | 0.345 | 0.224 |

**This directly resolves the open question these notes raised.** H7 (CPU, 10 epochs,
no LR warmup) found Prithvi showed no working label-efficiency advantage -- water IoU
identical across budgets, built_up actually dropping with more data. With the now-
verified GPU recipe, Prithvi's built_up IoU clearly increases with more data (0.152 ->
0.188 -> 0.284, the opposite direction from H7), and water is genuinely responsive to
budget (though non-monotonic, peaking at 50%), not flat. H7's negative finding was a
real artifact of under-training, not a real property of Prithvi -- the foundation
model does show a real, if imperfect, label-efficiency curve once actually trained
properly.

U-Net (from scratch) shows the clean, expected monotonic curve on all three classes at
every budget -- unsurprising for a model with no pretraining to fall back on. Prithvi
beats U-Net at every single budget/class combination except built_up at the 25%
budget, where U-Net edges it out (0.164 vs 0.152) -- the one place the foundation-model
story doesn't hold even directionally.

Clay (frozen encoder, small head) shows a different, real pattern: built_up IoU
slightly decreases with more data (0.248 -> 0.236 -> 0.224) and water stays
essentially flat (0.345 -> 0.349 -> 0.345, noise-level). A plausible honest
explanation, not confirmed further here: a small trainable head on a frozen encoder
may saturate its limited capacity quickly, so additional data past 25% does not
clearly help and may even shift the head slightly toward the majority class. Not
investigated further -- reported as a real, observed pattern.

No GPU memory issues or crashes across all 6 chained runs in one session (epoch
markers at stopping: 21, 99, 44, 80, 99, 30 -- consistent with different
models/budgets converging at genuinely different points, not a systematic failure).
